In [2]:
import re
from pathlib import Path

import pandas as pd

In [4]:
version = "eval_v2"
clfs = {"attn"}

paths = sorted(Path("output/long_seq").rglob(f"*/{version}/*/eval_table.csv"))
paths = [p for p in paths if p.parts[-2].split("__")[-1] in clfs]
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"T([0-9]+)_pt([0-9]+)_([0-9])", key)
    seq_length, pt, run = match.groups()
    table.insert(0, "run", int(run))
    table.insert(0, "pt", int(pt))
    table.insert(0, "seq_length", int(seq_length))
    table.insert(0, "key", key)
    tables.append(table)

paths = sorted(
    Path("../subcortical/output/subcortical").rglob(
        f"schaefer400_lr3e-4_*/{version}/*/eval_table.csv"
    )
)
paths = [p for p in paths if p.parts[-2].split("__")[-1] in clfs]
print(len(paths))
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"_([0-9])", key)
    (run,) = match.groups()
    table.insert(0, "run", int(run))
    table.insert(0, "pt", 4)
    table.insert(0, "seq_length", 16)
    table.insert(0, "key", key)
    tables.append(table)

state_table = pd.concat(tables, ignore_index=True)
print(state_table.shape)
state_table.head()

12
8
(70, 20)


,key,seq_length,pt,run,model,repr,clf,dataset,ckpt,epoch,lr,wd,hparam_id,hparam,split,loss,acc,acc_std,f1,f1_std
0,T16_pt16_1,16,16,1,schaefer400_mae,patch,attn,hcpya_task21,best,16,0.00213,0.05,36,"[7.1, 1.0]",train,0.000060,1.000000,0.000000,1.000000,0.000000
1,T16_pt16_1,16,16,1,schaefer400_mae,patch,attn,hcpya_task21,best,16,0.00213,0.05,36,"[7.1, 1.0]",validation,0.138051,0.976190,0.002368,0.970549,0.003179
2,T16_pt16_1,16,16,1,schaefer400_mae,patch,attn,hcpya_task21,best,16,0.00213,0.05,36,"[7.1, 1.0]",test,0.169561,0.973413,0.002244,0.968964,0.002904
3,T16_pt16_1,16,16,1,schaefer400_mae,patch,attn,nsd_cococlip,best,12,0.00129,0.05,33,"[4.3, 1.0]",train,2.081590,0.366852,0.002386,0.321826,0.002507
4,T16_pt16_1,16,16,1,schaefer400_mae,patch,attn,nsd_cococlip,best,12,0.00129,0.05,33,"[4.3, 1.0]",validation,2.516550,0.266519,0.005514,0.210514,0.005136


In [5]:
state_summary = state_table.loc[state_table["split"] == "test"].pivot_table(
    values=["acc", "acc_std"], index=["seq_length", "pt", "run", "repr", "clf"], columns="dataset"
)
state_summary = state_summary.loc[(slice(None), slice(None), slice(None), "patch", "attn")].dropna(
    axis=1, how="all"
)
state_summary

acc                   acc_std             
dataset           hcpya_task21 nsd_cococlip hcpya_task21 nsd_cococlip
seq_length pt run                                                    
16         4  1       0.973810     0.272542     0.002263     0.005209
              2       0.972817     0.277737     0.002331     0.005153
              3       0.971429     0.275325     0.002372     0.005251
              4       0.973214     0.274954     0.002301     0.005252
           16 1       0.973413     0.267718     0.002244     0.005031
              2       0.974008     0.279035     0.002247     0.005185
64         16 1       0.957937     0.254545     0.002989     0.005326
              2       0.954960     0.243599     0.002870     0.005155
              3       0.958929     0.257699     0.002716     0.005341
              4       0.957738     0.249907     0.002846     0.005253

In [6]:
DATASET_NAMES = {
    "abide_dx": "ABIDE Dx",
    "abide_age": "ABIDE Age",
    "abide_sex": "ABIDE Sex",
    "adhd200_dx": "ADHD200 Dx",
    "adhd200_sex": "ADHD200 Sex",
    "adni_ad_vs_cn": "ADNI Dx",
    "adni_sex": "ADNI Sex",
    "ppmi_dx": "PPMI Dx",
    "ppmi_sex": "PPMI Sex",
    "ppmi_age": "PPMI Age",
    "hcpya_rest1lr_gender": "HCP-YA Sex",
    "hcpya_rest1lr_age": "HCP-YA Age",
    "hcpya_rest1lr_neofacn": "HCP-YA NEO-N",
    "aabc_sex": "HCP-A Sex",
    "aabc_age": "HCP-A Age",
    "hcpya_task21": "HCP-YA Task21",
    "nsd_cococlip": "NSD COCO24",
}


def format_acc_std(acc, std):
    """Format accuracy and std as 'XX.XX ± YY.YY'"""
    if pd.isna(acc) or pd.isna(std):
        return "—"
    return f"{acc * 100:.1f} ± {std * 100:.1f}"
    # return f"{acc * 100:.1f} \mypm{{{std * 100:.1f}}}"

In [7]:
state_datasets = ["hcpya_task21", "nsd_cococlip"]

records = []

for (seq_length, pt, run), row in state_summary.iterrows():
    record = {"seq_length": seq_length, "pt": pt, "run": run}
    for ds in state_datasets:
        acc = row[("acc", ds)]
        std = row[("acc_std", ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "\mypm"))

|   seq_length |   pt |   run | HCP-YA Task21   | NSD COCO24   |
|-------------:|-----:|------:|:----------------|:-------------|
|           16 |    4 |     1 | 97.4 ± 0.2      | 27.3 ± 0.5   |
|           16 |    4 |     2 | 97.3 ± 0.2      | 27.8 ± 0.5   |
|           16 |    4 |     3 | 97.1 ± 0.2      | 27.5 ± 0.5   |
|           16 |    4 |     4 | 97.3 ± 0.2      | 27.5 ± 0.5   |
|           16 |   16 |     1 | 97.3 ± 0.2      | 26.8 ± 0.5   |
|           16 |   16 |     2 | 97.4 ± 0.2      | 27.9 ± 0.5   |
|           64 |   16 |     1 | 95.8 ± 0.3      | 25.5 ± 0.5   |
|           64 |   16 |     2 | 95.5 ± 0.3      | 24.4 ± 0.5   |
|           64 |   16 |     3 | 95.9 ± 0.3      | 25.8 ± 0.5   |
|           64 |   16 |     4 | 95.8 ± 0.3      | 25.0 ± 0.5   |


\begin{tabular}{rrrll}
\toprule
seq_length & pt & run & HCP-YA Task21 & NSD COCO24 \\
\midrule
16 & 4 & 1 & 97.4 \mypm 0.2 & 27.3 \mypm 0.5 \\
16 & 4 & 2 & 97.3 \mypm 0.2 & 27.8 \mypm 0.5 \\
16 & 4 & 3 & 97.1 \mypm 0.2 & 27.5 \mypm 0.5 \\
16 & 4 & 4 & 97.3 \mypm 0.2 & 27.5 \mypm 0.5 \\
16 & 16 & 1 & 97.3 \mypm 0.2 & 26.8 \mypm 0.5 \\
16 & 16 & 2 & 97.4 \mypm 0.2 & 27.9 \mypm 0.5 \\
64 & 16 & 1 & 95.8 \mypm 0.3 & 25.5 \mypm 0.5 \\
64 & 16 & 2 & 95.5 \mypm 0.3 & 24.4 \mypm 0.5 \\
64 & 16 & 3 & 95.9 \mypm 0.3 & 25.8 \mypm 0.5 \\
64 & 16 & 4 & 95.8 \mypm 0.3 & 25.0 \mypm 0.5 \\
\bottomrule
\end{tabular}



In [13]:
seq_lengths_pts = [(16, 4), (16, 16), (64, 16)]

records = []
for seq_length, pt in seq_lengths_pts:
    record = {"seq_length": seq_length, "pt": pt}
    for ds in state_datasets:
        acc = state_summary.loc[(seq_length, pt), ("acc", ds)].mean()
        std = state_summary.loc[(seq_length, pt), ("acc", ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "\mypm"))

|   seq_length |   pt | HCP-YA Task21   | NSD COCO24   |
|-------------:|-----:|:----------------|:-------------|
|           16 |    4 | 97.3 ± 0.1      | 27.5 ± 0.2   |
|           16 |   16 | 97.4 ± 0.0      | 27.3 ± 0.8   |
|           64 |   16 | 95.7 ± 0.2      | 25.1 ± 0.6   |
\begin{tabular}{rrll}
\toprule
seq_length & pt & HCP-YA Task21 & NSD COCO24 \\
\midrule
16 & 4 & 97.3 \mypm 0.1 & 27.5 \mypm 0.2 \\
16 & 16 & 97.4 \mypm 0.0 & 27.3 \mypm 0.8 \\
64 & 16 & 95.7 \mypm 0.2 & 25.1 \mypm 0.6 \\
\bottomrule
\end{tabular}



In [14]:
version = "eval_v2"
clfs = {"logistic"}

paths = sorted(Path("output/long_seq").rglob(f"*/{version}/*/eval_table.csv"))
paths = [p for p in paths if p.parts[-2].split("__")[-1] in clfs]
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"T([0-9]+)_pt([0-9]+)_([0-9])", key)
    seq_length, pt, run = match.groups()
    table.insert(0, "run", int(run))
    table.insert(0, "pt", int(pt))
    table.insert(0, "seq_length", int(seq_length))
    table.insert(0, "key", key)
    tables.append(table)

paths = sorted(
    Path("../subcortical/output/subcortical").rglob(
        f"schaefer400_lr3e-4_*/{version}/*/eval_table.csv"
    )
)
paths = [p for p in paths if p.parts[-2].split("__")[-1] in clfs]
print(len(paths))
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"_([0-9])", key)
    (run,) = match.groups()
    table.insert(0, "run", int(run))
    table.insert(0, "pt", 4)
    table.insert(0, "seq_length", 16)
    table.insert(0, "key", key)
    tables.append(table)


trait_table = pd.concat(tables, ignore_index=True)
print(trait_table.shape)
trait_table.head()

36
24
(12120, 17)


,key,seq_length,pt,run,model,repr,clf,dataset,trial,C,split,acc,acc_std,f1,f1_std,bacc,bacc_std
0,T16_pt16_1,16,16,1,schaefer400_mae,patch,logistic,aabc_age,NaN,0.000774,train,0.523622,0.021880,0.512945,0.022309,0.521640,0.021792
1,T16_pt16_1,16,16,1,schaefer400_mae,patch,logistic,aabc_age,NaN,0.000774,test,0.307692,0.059193,0.276989,0.052236,0.295330,0.057483
2,T16_pt16_1,16,16,1,schaefer400_mae,patch,logistic,aabc_age,1.0,0.046416,train,0.755906,0.018733,0.755442,0.018877,0.756921,0.018682
3,T16_pt16_1,16,16,1,schaefer400_mae,patch,logistic,aabc_age,1.0,0.046416,test,0.423077,0.060256,0.403513,0.062004,0.420788,0.060063
4,T16_pt16_1,16,16,1,schaefer400_mae,patch,logistic,aabc_age,2.0,0.000774,train,0.525591,0.021099,0.510276,0.021823,0.524359,0.020963


In [15]:
def sem(x):
    return x.std() / (len(x) ** 0.5)


trait_repr = "patch"

trait_summary = trait_table.query("split == 'test' and trial > 0").pivot_table(
    values=["acc", "bacc"],
    index=["seq_length", "pt", "run", "repr", "clf"],
    columns="dataset",
    aggfunc=["mean", sem],
)
trait_summary = trait_summary.loc[
    (slice(None), slice(None), slice(None), trait_repr, "logistic")
].dropna(axis=1, how="all")
trait_summary

mean                                               \
                        acc                                                
dataset            aabc_age  aabc_sex  abide_dx adhd200_dx adni_ad_vs_cn   
seq_length pt run                                                          
16         4  1    0.442115  0.723818  0.620323   0.589231      0.730244   
              2    0.442308  0.710364  0.624032   0.582308      0.731707   
              3    0.451154  0.723818  0.636290   0.587231      0.747317   
              4    0.434808  0.720909  0.637258   0.594923      0.746829   
           16 1    0.431538  0.729636  0.617661   0.609846      0.733902   
              2    0.431923  0.718545  0.646694   0.600462      0.720000   
64         16 1    0.442692  0.738909  0.630726   0.612000      0.750488   
              2    0.438269  0.744545  0.621048   0.597538      0.734634   
              3    0.441923  0.729455  0.627419   0.606769      0.745122   
              4    0.440385  0.747818  0.621694   0.605231      0.747805   

                                                                    ...  \
                               bacc                                 ...   
dataset           ppmi_dx  aabc_age  aabc_sex  abide_dx adhd200_dx  ...   
seq_length pt run                                                   ...   
16         4  1    0.6506  0.440250  0.712643  0.614165   0.569040  ...   
              2    0.6359  0.440366  0.699918  0.617815   0.564870  ...   
              3    0.6379  0.449766  0.711298  0.631213   0.564112  ...   
              4    0.6526  0.432308  0.708920  0.631812   0.574821  ...   
           16 1    0.6493  0.429783  0.719416  0.612337   0.593793  ...   
              2    0.6351  0.429522  0.706827  0.641628   0.583682  ...   
64         16 1    0.6307  0.439190  0.727385  0.624501   0.591733  ...   
              2    0.6553  0.436454  0.732106  0.613739   0.579291  ...   
              3    0.6371  0.439540  0.719443  0.621077   0.590439  ...   
              4    0.6518  0.438613  0.738465  0.614359   0.587611  ...   

                        sem                                               \
                        acc                                         bacc   
dataset            abide_dx adhd200_dx adni_ad_vs_cn   ppmi_dx  aabc_age   
seq_length pt run                                                          
16         4  1    0.004031   0.005970      0.006131  0.004512  0.007247   
              2    0.004167   0.006089      0.006064  0.004030  0.006950   
              3    0.004311   0.005880      0.006509  0.004591  0.006144   
              4    0.004219   0.005454      0.005673  0.005032  0.006196   
           16 1    0.004261   0.005163      0.006212  0.004482  0.006448   
              2    0.004162   0.004803      0.006378  0.004270  0.006517   
64         16 1    0.003702   0.005825      0.006033  0.004169  0.007072   
              2    0.003842   0.005983      0.006067  0.004838  0.006797   
              3    0.004139   0.005930      0.006058  0.004652  0.006431   
              4    0.004020   0.005753      0.005906  0.004300  0.006311   

                                                                          
                                                                          
dataset            aabc_sex  abide_dx adhd200_dx adni_ad_vs_cn   ppmi_dx  
seq_length pt run                                                         
16         4  1    0.005979  0.004091   0.006044      0.007349  0.004444  
              2    0.007111  0.004145   0.006274      0.007892  0.004356  
              3    0.005960  0.004303   0.005860      0.008667  0.004722  
              4    0.006303  0.004349   0.005497      0.007526  0.005246  
           16 1    0.005566  0.004218   0.005294      0.007823  0.004773  
              2    0.005838  0.004159   0.004931      0.007831  0.004445  
64         16 1    0.006054  0.003656   0.005929      0.007561  0.004430  
              2  

In [16]:
trait_datasets = ["abide_dx", "adhd200_dx", "adni_ad_vs_cn", "ppmi_dx", "aabc_age", "aabc_sex"]
trait_metric = "bacc"

records = []

for (seq_length, pt, run), row in trait_summary.iterrows():
    record = {"seq_length": seq_length, "pt": pt, "run": run}
    for ds in trait_datasets:
        acc = row[("mean", trait_metric, ds)]
        std = row[("sem", trait_metric, ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "\mypm"))

|   seq_length |   pt |   run | ABIDE Dx   | ADHD200 Dx   | ADNI Dx    | PPMI Dx    | HCP-A Age   | HCP-A Sex   |
|-------------:|-----:|------:|:-----------|:-------------|:-----------|:-----------|:------------|:------------|
|           16 |    4 |     1 | 61.4 ± 0.4 | 56.9 ± 0.6   | 61.1 ± 0.7 | 61.2 ± 0.4 | 44.0 ± 0.7  | 71.3 ± 0.6  |
|           16 |    4 |     2 | 61.8 ± 0.4 | 56.5 ± 0.6   | 60.1 ± 0.8 | 60.2 ± 0.4 | 44.0 ± 0.7  | 70.0 ± 0.7  |
|           16 |    4 |     3 | 63.1 ± 0.4 | 56.4 ± 0.6   | 63.8 ± 0.9 | 59.7 ± 0.5 | 45.0 ± 0.6  | 71.1 ± 0.6  |
|           16 |    4 |     4 | 63.2 ± 0.4 | 57.5 ± 0.5   | 63.2 ± 0.8 | 61.6 ± 0.5 | 43.2 ± 0.6  | 70.9 ± 0.6  |
|           16 |   16 |     1 | 61.2 ± 0.4 | 59.4 ± 0.5   | 59.9 ± 0.8 | 61.2 ± 0.5 | 43.0 ± 0.6  | 71.9 ± 0.6  |
|           16 |   16 |     2 | 64.2 ± 0.4 | 58.4 ± 0.5   | 58.3 ± 0.8 | 59.6 ± 0.4 | 43.0 ± 0.7  | 70.7 ± 0.6  |
|           64 |   16 |     1 | 62.5 ± 0.4 | 59.2 ± 0.6   | 63.8 ± 0.8 | 60.0 ± 0.4 | 43

In [18]:
records = []

for seq_length, pt in seq_lengths_pts:
    record = {"seq_length": seq_length, "pt": pt}
    for ds in trait_datasets[4:]:
        acc = trait_summary.loc[(seq_length, pt), ("mean", trait_metric, ds)].mean()
        std = trait_summary.loc[(seq_length, pt), ("mean", trait_metric, ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    for ds in state_datasets:
        acc = state_summary.loc[(seq_length, pt), ("acc", ds)].mean()
        std = state_summary.loc[(seq_length, pt), ("acc", ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
# print(result_fmt.to_latex(index=False).replace("±", "\mypm"))
print(re.sub("± ([.0-9]+)", r"\\mypm{\1}", result_fmt.to_latex(index=False)))

|   seq_length |   pt | HCP-A Age   | HCP-A Sex   | HCP-YA Task21   | NSD COCO24   |
|-------------:|-----:|:------------|:------------|:----------------|:-------------|
|           16 |    4 | 44.1 ± 0.7  | 70.8 ± 0.6  | 97.3 ± 0.1      | 27.5 ± 0.2   |
|           16 |   16 | 43.0 ± 0.0  | 71.3 ± 0.9  | 97.4 ± 0.0      | 27.3 ± 0.8   |
|           64 |   16 | 43.8 ± 0.1  | 72.9 ± 0.8  | 95.7 ± 0.2      | 25.1 ± 0.6   |
\begin{tabular}{rrllll}
\toprule
seq_length & pt & HCP-A Age & HCP-A Sex & HCP-YA Task21 & NSD COCO24 \\
\midrule
16 & 4 & 44.1 \mypm{0.7} & 70.8 \mypm{0.6} & 97.3 \mypm{0.1} & 27.5 \mypm{0.2} \\
16 & 16 & 43.0 \mypm{0.0} & 71.3 \mypm{0.9} & 97.4 \mypm{0.0} & 27.3 \mypm{0.8} \\
64 & 16 & 43.8 \mypm{0.1} & 72.9 \mypm{0.8} & 95.7 \mypm{0.2} & 25.1 \mypm{0.6} \\
\bottomrule
\end{tabular}

